In [1]:
batch_number = 1

In [2]:
import os
import re
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

from bs4 import BeautifulSoup
from fuzzywuzzy import fuzz

# Environment Variables
from dotenv import load_dotenv

# Llama Model
from langchain_community.llms import LlamaCpp # Llm Handler
from langchain.prompts import PromptTemplate # Prompt
from langchain_core.output_parsers import StrOutputParser # Parser

# NER Model
import spacy

# Google Maps API
import requests
import googlemaps

# Topic Modeling
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## Running Llama 3.1 8B

In [3]:
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article. Give your answer in the following format:
    1. If there is one, the city the article is talking about. Otherwise, state that it can't be located. 
    2. The specific place within the city you got if you found one.
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision.
     
    If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)


In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 33 key-value pairs and 292 tensors from ./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Meta Llama 3.1 8B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Meta-Llama-3.1
llama_model_loader: - kv   5:                         general.size_label str              = 8B
llama_model_loader: - kv   6:                            general.license str              = llama3.1
llama_model_l

In [6]:
chain = prompt | llm | output_parser

## NER Model

In [7]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [8]:
# Load environment variables
load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [9]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [10]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [11]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

## Pipeline Entry Point

### Load and Preprocess Data Set

In [12]:
# Load the data set
data_path = f"./data_set/batch_{batch_number}.csv"
full_df = pd.read_csv(data_path)

In [13]:
def partial_df(df):
    columns = df.columns

    required_columns = {'Byline': 'author',
                        'Body': 'body', 
                        "Headline": 'hl1', 
                        'Publish Date': 'pub_date', 
                        'Publisher': 'pub_name', 
                        'Paths': 'link'}
    
    partial_df = pd.DataFrame()
    for column in required_columns:
        if column in columns:
            partial_df[required_columns[column]] = df[column]
        else: 
            print(f"Error: Column {column} not found in the data set")
            return None
    
    # Make 'tagging' column be the id column
    tagging_col = df.get('Tagging')
    partial_df.insert(0, '_id', tagging_col)

    # Drop rows where at least one of the specified columns is empty
    columns_to_check = ['_id', 'hl1', 'body']
    partial_df = partial_df.dropna(subset=columns_to_check, how='all')

    # Drop empty rows too
    partial_df = partial_df[~partial_df['body'].apply(lambda x: isinstance(x, float))]
    partial_df = partial_df[~partial_df['hl1'].apply(lambda x: isinstance(x, float))]

    return partial_df

def clean_df(partial_df):
    cleaned_df = partial_df.copy()
    
    # Clean the html
    func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
    cleaned_df['body'] = partial_df['body'].progress_apply(func_clean_html)
    cleaned_df['hl1'] = cleaned_df['hl1'].progress_apply(func_clean_html)

    # Clean with Regex
    func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
    cleaned_df['body'] = cleaned_df['body'].progress_apply(func_clean_regex)
    cleaned_df['hl1'] = cleaned_df['hl1'].progress_apply(func_clean_regex)

    return cleaned_df


### Location Utility Functions

In [14]:
# Normalize a location string
def normalize_location(location):
    location = location.lower().strip() 
    
    # Remove leading terms
    leading_terms = ["a", "an", "the"]

    for term in leading_terms:
        if location.startswith(term + " "):
            location = location[len(term) + 1:]

    # Remove punctuation
    location = re.sub(r'[^\w\s]', '', location)
    
    # Expand common abbreviations
    abbreviation_map = {
        "us": "united states",
        'co': 'company',
        'pd': 'police department',
        'wgbh': 'gbh',
        'cfa': 'the harvard-smithsonian center for astrophysics',
        'pbs': 'public broadcasting service',
        'ap': 'associated press',
        'aim': 'aim (alternative investment market)',
        '&': 'and',
        'npr': 'national public radio',
        'doj': 'the justice department',
        'cdc': 'the centers for disease control and prevention',
        'ssa': 'social security administration',
        'doe': 'the energy department',
        'cbpp': 'the center on budget and policy priorities',
        'mbta': 'massachusetts bay transportation authority',
        't': 'massachusetts bay transportation authority',
        'globe': 'boston globe',
        'senate': 'capitol',
        'congress': 'capitol',
        'legislature': 'capitol',
        'justice': 'department of justice',
        'house': 'u.s. house of representatives',
        'oval office': 'the white house',
        'dese': 'department of elementary and secondary education',
        'bps': 'boston public schools',
        'dhs': 'the department of homeland security',
        'fed': 'federal reserve',
        'fbi': 'the federal bureau of investigation',
        'epa': 'the environmental protection agency',
        'cdc': 'the centers for disease control and prevention',
        'nar': 'national association of realtors',
        'adl': 'anti-defamation league',
        'cbp': 'customs and border protection',
        'cia': 'central intelligence agency',
        'fda': 'food and drug administration',
        'dep': 'department of environmental protection',
        'un': 'united nations',
        'faa': 'federal aviation administration',
        'ntsb': 'national transportation safety board',
        'dua': 'department of unemployment assistance',
        'necn': 'new england cable news',
        'nar': 'national association of realtors',
        'who': 'world health organization',
        'irap': 'international refugee assistance project',
        'ncaa': 'national collegiate athletic association',
        'council': 'city council',
        'dph': 'the department of public health',
        'usda': 'the us department of agriculture',
        'bpr': 'boston public radio',
        'blm': 'black lives matter',
        'irs': 'internal revenue service',
        'necn': 'new england cable news',
        'wh': 'white house',
        'gop': 'the republican party',
        'ps': 'public schools',
        'ma dese': 'the massachusetts department of elementary and secondary education',
    }
    for abbr, full in abbreviation_map.items():
        location = re.sub(r'\b' + abbr + r'\b', full, location)
    
    # TODO: (maybe) Handle synonyms, variants
    # TODO: (maybe) Handle country/state/city abbreviations   
    
    # Remove extra spaces
    location = re.sub(r'\s+', ' ', location)
    
    return location

In [15]:
# Check if two locations are the same
def are_same_location(loc1, loc2, threshold=80):
    if loc1 == loc2:
        return True
    
    if loc1 in loc2 or loc2 in loc1:
        return True
    
    similarity = fuzz.token_set_ratio(loc1, loc2)
    return similarity >= threshold

# Combine two locations if they are the same
def combine_locations(location, locations):
    if (locations is None or len(locations) == 0):
        return [location]
    
    # check for news subdomains
    if "gbh" in location:
        location = "gbh"
    elif "npr" in location:
        location = "npr"

    new_locs = []

    for loc in locations:
        if are_same_location(location, loc):
            # Replace the existing location if the new one is longer
            if len(location) > len(loc):
                new_locs.append(location)
            else:
                new_locs.append(loc)
        else:
            new_locs.append(loc)

    new_locs.append(location)

    return new_locs

In [16]:
# Check if a location can be added to the list of locations
def can_add_location(location):
    unwanted_entities = load_cache("./data_prod/unwanted_locations.json")

    if location in unwanted_entities:
        return False
    elif invalid_location(location):
        return False
    else:
        return True

# Check if a location is unwanted
def invalid_location(location):
    # List of common unwanted entity types
    unwanted_places = [
        r'\bstreet\b', 
        r'\bsquare\b', 
        r'\bavenue\b', 
        r'\bboulevard\b',
        r'\broad\b', 
        r'\blane\b', 
        r'\bdrive\b', 
        r'\bdriveway\b',
        r'\bhighway\b',
        r'\bfreeway\b'
    ]
    
    # Create a combined regex pattern
    pattern = re.compile('|'.join(unwanted_places))
    
    # Check if the location matches any unwanted entity type
    if pattern.search(location):
        return True
    return False

## Explicit Pass

### Locate based on Explicit Mention of Locations on Title

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [17]:
known_title_locs_path = "./data_prod/known_locations.json"
known_title_locs = load_cache(known_title_locs_path)
known_locations = known_title_locs.keys()

unwanted_entities_path = "./data_prod/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [18]:
# If a location is in the title, use that as the article's location
def get_title_entities(header):
    # Look through the header for known locations
    locations_list = []
    for location in known_locations:
        loc = normalize_location(location)
        if (loc in header and can_add_location(loc)):
            locations_list.append(loc)
        
        if (len(locations_list) == 5):
            break
    
    if (len(locations_list) == 0):
        return None
    else:
        return locations_list

## NER Pass

### Identify main 5 locations of the article with NER

In [19]:
from collections import Counter

# Get the top 5 most common locations
def get_main_5(facilities, organizations):

    fac_freq = Counter(facilities)
    org_freq = Counter(organizations)

    top_fac = fac_freq.most_common(1) if facilities else []
    top_org = org_freq.most_common(1) if organizations else []

    combined = facilities + organizations
    combined_freq = Counter(combined)

    if top_fac:
        combined_freq.pop(top_fac[0][0], None)
    if top_org:
        combined_freq.pop(top_org[0][0], None)
    
    top_combined = combined_freq.most_common(3)

    top_entities = top_fac + top_org + top_combined
    
    top_entities = [entity[0] for entity in top_entities]

    return top_entities

In [20]:
def add_entity(entity, valid_list):
    loc = normalize_location(entity)
    if (can_add_location(loc)):
        valid_list = combine_locations(loc, valid_list)
    
    return valid_list

In [21]:
# Return all valid facilities and organizations found
def get_valid_entities(entities):
    valid_facs = []
    valid_orgs = []

    if (entities is None or len(entities) == 0):
        return None

    for entity in entities:
        if (entity.label_ == "FAC"):
            valid_facs = add_entity(entity.text, valid_facs)
        elif (entity.label_ == "ORG"):
            valid_orgs = add_entity(entity.text, valid_orgs)

    
    valid_entities = get_main_5(valid_facs, valid_orgs)
    
    if (len(valid_entities) == 0):
        return None
    else:
        return valid_entities
        

In [22]:
# Run NER on the body of the article and return first valid facility
def run_NER(text, truncate=True):
    if (truncate): # Truncate the text to the first 500 words
        text = ' '.join(text.split()[:500])

    if (text == None or text == ""):
        return None
    
    try:
        entities = nlp(text).ents
        valid_entities = get_valid_entities(entities)
        return valid_entities
        
    except Exception as error:
        print(error)
        return None

In [23]:
def process_NER(article):
    """
    Process the NER on the body of the article and return all valid facilities and organizations found. If 'truncate' is true, then we get the first 500 words.
    """
    if (article['Explicit_Pass'] != None): 
        return None
    else:
        valid_entities = run_NER(article['body'])
        return valid_entities

### Llama Prediction

In [24]:
# Run the LLM model on the title and body of the article.
def run_llm(title, body):
    try:
        llama_prediction = chain.invoke({"headline": title, "body": body})
        return llama_prediction
    except Exception as error:
        print("Failed running the LLM: ", error)
        return None

In [25]:
def process_LLM(article):
    """
    Try to predict the location of the article using the LLM model. Then run NER on prediction to obtain locations.
    """

    # If the article does not have an explicit location or NER location, run LLM
    if (article['Explicit_Pass'] != None):
        return None
    elif (article['NER_Pass'] != None):
        return None
    
    else:
        llama_prediction = run_llm(article['hl1'], article['body'])
        valid_entities = run_NER(llama_prediction)
        return valid_entities

## Get Locations From Passes

In [26]:
# Get all locations from the article
def getAllLocations(article):
    locations_list = []
    for key in ['Explicit_Pass', 'NER_Pass', 'LLM_Pass']:
        location = article[key]
        if location is not None:
            locations_list.extend(location)
            break
    
    if len(locations_list) == 0:
        return None
    else:
        return locations_list

## Get the Location Coordinates

In [27]:
known_locations_path = "./data_prod/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [28]:
# Get the coordinates of the location
def getCoordinates(location): 
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [29]:
def getAllCoordinates(locations):
    coordinates_list = []

    if (locations == None): return None
    
    for location in locations:
        coordinates = getCoordinates(location)
        if coordinates is not None:
            coordinates_list.append(coordinates)
    return coordinates_list

## Geocode locations

In [30]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    print("API call failed for: " + location + " with coordinates" + str(coordinates))
    return None, None  # Return this if API call failed or no tracts found

In [31]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County

In [32]:
def getAllGeocodes(locations):
    tracts = []
    counties = []

    if (locations == None): return None, None
    
    for location in locations:
        Tract, County = geocode(location)
        tracts.append(Tract)
        counties.append(County)

    return tracts, counties

## Get Neighborhoods

In [33]:
neigh_tract_dict = {
	"Fenway" : ["010103", "010104", "010204", "010408", "010404", "010403", "981501", "010405", "010206", "010205"],
	"Downtown": ["030302", "070202", "070102", "030301", "070104", "070103", "070201"],
	"Beacon Hill": ["020200", "020302", "020101", "981700"],
	"Dorchester" : [
	"092400", "091400", "090300", "091800", "092300", "100601", "090901", 
	"100400", "090100", "091001","090200", "100200", "091700", "092200", "090700",
	"091500", "091300", "100300", "100100", "092000", "100500", "100800", "100603",
	"091200", "100700", "092101", "091900", "091600", "091100"
	],
	"Mattapan": ["100900", "101002", "101102", "981100", "101001","101101"],
	"Jamaica Plain": [
	"120103", "981800", "110105", "120600", "120700", "120301", "081200", "120105","081101",
	"981000", "120500", "120104", "120201", "110106", "081301", "120400"
	],
	"Roslindale": ["110502", "110104", "110501", "110401", "140106", "110301", "110607", "110403","110201"],
	"Roxbury": [
	"081500", "080500", "070801", "080100", "081800", "980300", "082000", "080601", "081700", "080300",
	"090600", "081400", "090400", "070901", "082100", "081900", "081302","080401"
	],
	"West End": ["020304", "020301", "020305"],
	"Longwood": ["010300", "081001"],
	"South Boston": ["061101", "060700", "060101", "061201", "061000", "060800", "981201", "060200", "061202", "060400", "061203", "060301", "060601", "060501"],
	"Back Bay": ["010702", "010701", "010802", "010801", "010500", "010600"],
	"Charlestown": ["040100", "040300", "040401", "040600", "040801", "040200"],
	"Allston": ["000604", "000804", "000703", "000704", "000806", "000101", "000807", "000701", "000805"],
	"Hyde Park": ["140107", "140201", "140105", "980700", "140300", "140202", "140400", "140102"],
	"East Boston": ["050500", "050600", "981502", "050101", "981300", "050901", "050300", "050700", "050400", "051000", "981600", "051200", "050200", "051101"],
	"South End": ["070301", "070302", "070502", "070501", "071101", "070600", "070700", "070902", "070802", "071201", "070402"],
	"West Roxbury": ["980900", "130406", "981900", "130404", "110601", "130300", "130402", "130200", "130101"],
	"South Boston Waterfront": ["981202", "060602", "060603", "061204", "060604"],
	"North End": ["030200", "030100", "030500", "030400"],
	"Cambridge": ["354300", "354200", "353102", "353600", "352300", "354100", "359400", "353300", "353700", "353200", 
	"354601", "355000", "354602", "354000", "354901", "354902", "353900", "354700", "352102", "354500", "354800", "352600", 
	"354400", "353101", "352900", "353000", "352101", "353800", "352500", "352400", "352700", "352200", "352800", 
    "365100", "361300"
  	],
	"Chelsea": ["160400", "160103", "160102", "160300", "160601", "160602", "160501", "160502", "160200"],
	

}

In [34]:
tract_map_path = "./data_prod/tract_map.json"
tract_map = load_cache(tract_map_path)

In [35]:
if (tract_map == {}):
    for neigh, tracts in neigh_tract_dict.items():
        for tract in tracts:
            tract_map[tract] = neigh

    save_cache_to_file(tract_map, tract_map_path)

In [36]:
def getAllNeighborhoods(articles):
    tracts = articles['tracts']

    neighborhoods = []

    if (tracts == None): return None

    for tract in tracts:
        if (tract == None): continue

        neighborhood = tract_map.get(tract)
        if neighborhood is not None:
            neighborhoods.append(neighborhood)
        else:
            neighborhoods.append("Unknown Neighborhood")

    return neighborhoods

## Full Location Pipeline

In [37]:
def geolocate_articles(df):
    """
    Processes the dataaframe given by func. Does Entity Recognition and Geolocation on articles.
    
    Parameters
    ----
    df: The pandas dataframe that geolocation is being done on.

    Returns
    ---- 
    Returns a Dataframe of geolocated articles
    """
    try: 

        ### Explicit Mention Pass ###
        df["Explicit_Pass"] = df["hl1"].progress_apply(get_title_entities)

        ### NER Direct Pass ### 
        df["NER_Pass"] = df.progress_apply(process_NER, axis=1) # Automatically Truncates and performs NER on first 500 words
                
        ### Llama + NER Inference Pass ###
        df['LLM_Pass'] = df.progress_apply(process_LLM, axis=1) # Also truncates to 500 words
        
        # Extract Locations from Passes
        df['locations'] = df.progress_apply(getAllLocations, axis=1)

        # Get the Coordinates for the Locations
        df['coordinates'] = df['locations'].progress_apply(getAllCoordinates)

        # Geocode the Coordinates (Get the Tract and County)
        df[['tracts', 'counties']] = df['locations'].progress_apply(getAllGeocodes).apply(pd.Series)

        # Get the Neighborhoods
        df["neighborhoods"] = df.progress_apply(getAllNeighborhoods, axis=1)

        # Drop the rows that are missing information
        df = df.dropna(subset=["locations", "coordinates", "tracts", "counties", "neighborhoods"]) # Clean the rows that are missing information
        
        return df
    except Exception as e: 
        print(f"[Fatal Error] geolocate_articles() ran into an Error! Data is not saved!\nRaw Error:{e}")
        raise Exception(f"FATAL ERROR {e}")
    return

## Topic Modeling

## OpenAI Client

In [38]:
openai_key = os.getenv("OPENAI_API_KEY") 
client = OpenAI(
    api_key= openai_key,
)

In [39]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

## Taxonomy Lists

Content Taxanomy

In [40]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./data_prod/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

703


Selected Taxonomy List

In [41]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./data_prod/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [42]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./data_prod/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

In [43]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

### Full Topic Modeling Pipeline

In [44]:
def topic_modeling(df):
    """
    Processes dataframe and passes it to output topic labels. Does Topic Modeling task on articles.
    
    Parameters
    ----
    df: The pandas dataframe that topic modeling is being done on.

    Returns
    ---- 
    Returns a Dataframe of Topic Modeling articles
    """
    try:
        df['topic_model_body'] = df['body'].progress_apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
        df['tokens'] = df['topic_model_body'].progress_apply(lambda x: x.split())
        df['tokens'] = df['tokens'].progress_apply(truncate)
        df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

        # Find most similar taxonomy (out of all toipcs) to news body
        closest_topic_list_all = []
        for index, row in df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

            # Find the index of the topic with the highest similarity
            closest_topic_index = np.argmax(similarities)

            # Retrieve the closest topic embedding
            closest_topic = all_topics_list[closest_topic_index]
            closest_topic_list_all.append(closest_topic)
        df['closest_topic_all'] = closest_topic_list_all

        # Find most similar taxonomy (out of 230 selected topics) to news body
        closest_topic_list_selected = []
        for index, row in df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

            # Find the index of the topic with the highest similarity
            closest_topic_index = np.argmax(similarities)

            # Retrieve the closest topic embedding
            closest_topic = selected_topics_list[closest_topic_index]
            closest_topic_list_selected.append(closest_topic)

        df['closest_topic_selected'] = closest_topic_list_selected

        client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
        client_topic_list = client_taxonomy_df['label'].to_list()
        similarity_arr = []

        closest_topic_list_client = []
        for index, row in df.iterrows():
            target_embedding = row['ada_embedding']
            similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
            
            if max(similarities) > 0.25:    
                closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
                closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
                closest_topic_list_client.append([closest_topic])
            else:
                closest_topic_list_client.append(['Other'])
            similarity_arr.append(max(similarities))
            
        df['openai_labels'] = closest_topic_list_client

        return df
    except Exception as e: # Loop inbounded error
        print(f"[Error] topic_modeling() ran into an error! \n[Raw Error]: {e}")
        raise

## Running Everything

In [45]:
# Process articles in batches of 100
def process_articles(articles_df):
    results_df = pd.DataFrame()
    batch_size = 100
    batch_count = 1 + articles_df.shape[0] // batch_size
    total_count = 0
    for batch in range(0, articles_df.shape[0], batch_size):
        print(f"[INFO] Processing batch {batch_count} of {batch_count}")
        articles = articles_df[batch:batch + batch_size].copy()

        # Cleaning the articles
        print(f"[INFO] Cleaning Data")
        partial = partial_df(articles)
        if partial is None:
            print(f"[INFO] No articles passed the cleaning pipeline")
            continue
        cleaned_articles = clean_df(partial)

        # Conduct Entity Recognition
        print(f"[INFO] Processing through Geolocation Pipeline")
        processing_df = geolocate_articles(cleaned_articles)

        passes = ['Explicit_Pass', 'NER_Pass', 'LLM_Pass']
        for pass_ in passes:
            count = processing_df[pass_].notna().sum()
            print(f"[INFO] {count} articles passed {pass_}")
        
        if (processing_df.empty):
            print(f"[INFO] No articles passed the geolocation pipeline")
            continue

        print(f"[INFO] Processing through Topic Modeling Pipeline")
        topic_df = topic_modeling(processing_df)

        print("[INFO] Formatting Data")
        packaged_data_df = topic_df.drop(columns=[
            'Explicit_Pass', 
            'NER_Pass', 
            'LLM_Pass', 
            'topic_model_body',
            'tokens',
            'ada_embedding',
            'closest_topic_all',
            'closest_topic_selected',
        ])

        print(f"[INFO] Batch Complete! Recognized {packaged_data_df.shape[0]} articles")

        # Transform dataframe to JSON
        
        packaged_data_df.to_csv(f"./data_results/group_{batch_number}_batch_{batch}.csv", index=False)
        total_count += packaged_data_df.shape[0]
        results_df = pd.concat([results_df, packaged_data_df])

    print(f"[INFO] Inference Pipeline Complete! Total Articles Processed: {total_count}")
    return results_df


In [46]:
results_df = process_articles(full_df)

[INFO] Processing batch 1 of 1
[INFO] Cleaning Data


100%|██████████| 2/2 [00:00<00:00, 1744.72it/s]


[INFO] Processing through Geolocation Pipeline


100%|██████████| 2/2 [00:00<00:00, 269.12it/s]


[INFO] 0 articles passed Explicit_Pass
[INFO] 2 articles passed NER_Pass
[INFO] 0 articles passed LLM_Pass
[INFO] Processing through Topic Modeling Pipeline


100%|██████████| 2/2 [00:00<?, ?it/s]


[INFO] Formatting Data
[INFO] Batch Complete! Recognized 2 articles
[INFO] Inference Pipeline Complete! Total Articles Processed: 2


In [47]:
results_df.head(10)

,_id,author,body,hl1,pub_date,pub_name,link,locations,coordinates,tracts,counties,neighborhoods,openai_labels
0,0000016a-3bcb-d661-af7b-7bff0d200001,Brian O'Donovan,The above is continuous stream. If interested ...,Celtic Playlist For Easter,Mon Mar 29 15:22:35 EDT 2021,Publisher,/music/celtic/2019/04/20/a-celtic-playlist-for...,"[celtic sojourn, anjali quartet]","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100]","[017, 017]","[Cambridge, Cambridge]",[Arts & Culture]
1,0000016a-f0d5-dbfd-a56f-f4dfc34c0001,Brian O'Donovan,Click above for the audio of special segment o...,Songs Of War And Remembrance Memorial Day,Sat May 29 01:00:45 EDT 2021,Publisher,/music/celtic/2019/05/25/songs-of-war-and-reme...,"[gbh, spotify, united states academy band, cra...","[[-71.3824374, 42.4072107], [-71.3824374, 42.4...","[365100, 365100, 365100, 365100]","[017, 017, 017, 017]","[Cambridge, Cambridge, Cambridge, Cambridge]",[Obituaries]


In [49]:
results_df.to_csv(f"./data_results/group_{batch_number}_results.csv", index=False)

In [48]:
# df.to_csv(f"./results/full_benchmark_{sample_count}_samples_trial_{trial}.csv")
# time_df.to_csv(f"./results/full_benchmark_times_{sample_count}_samples_trial_{trial}.csv")